In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from matplotlib.animation import FuncAnimation, PillowWriter
# 1️⃣ Create synthetic dataset
X, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
# 2️⃣ Range of eps values for DBSCAN
eps_values = np.linspace(0.1, 1.0, 10)
all_labels = []
# 3️⃣ Metrics storage
sil_scores = []
calinski_scores = []
# 4️⃣ Apply DBSCAN for each eps value
for eps in eps_values:
    db = DBSCAN(eps=eps, min_samples=5)
    labels = db.fit_predict(X)
    all_labels.append(labels)

    # Compute metrics (only if more than 1 cluster)
    if len(set(labels)) - (1 if -1 in labels else 0) > 1:
        sil_scores.append(silhouette_score(X, labels))
        calinski_scores.append(calinski_harabasz_score(X, labels))
    else:
        sil_scores.append(np.nan)
        calinski_scores.append(np.nan)
# 5️⃣ Prepare animation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
def update(frame_idx):
    ax1.clear()
    labels = all_labels[frame_idx]
    eps = eps_values[frame_idx]
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    # Plot clustering result
    ax1.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=25)
    ax1.set_title(f"DBSCAN Clustering\n eps={eps:.2f}, clusters={n_clusters}")
    ax1.set_xlabel("Feature 1")
    ax1.set_ylabel("Feature 2")
    ax1.set_xlim(X[:,0].min()-0.2, X[:,0].max()+0.2)
    ax1.set_ylim(X[:,1].min()-0.2, X[:,1].max()+0.2)
    # Plot metrics so far
    ax2.clear()
    ax2.plot(eps_values[:frame_idx+1], sil_scores[:frame_idx+1], label="Silhouette Score", marker='o')
    ax2.plot(eps_values[:frame_idx+1], calinski_scores[:frame_idx+1], label="Calinski-Harabasz", marker='x')
    ax2.set_xlabel("eps value")
    ax2.set_ylabel("Score")
    ax2.set_title("Clustering Metrics")
    ax2.legend()
    ax2.grid(True)
ani = FuncAnimation(fig, update, frames=len(eps_values), interval=500, repeat=False)
# 6️⃣ Save as GIF
gif_path = "dbscan_eps_sweep_with_metrics.gif"
ani.save(gif_path, writer=PillowWriter(fps=2))
print(f"GIF saved to {gif_path}")